# Middleware

# 1.预构建中间件

## · PII中间件

In [1]:
from langchain.agents.middleware import PIIMiddleware, AgentMiddleware, hook_config
from langchain.agents.middleware.types import StateT
from langchain.messages import HumanMessage
from langchain.agents import create_agent
import re

from langgraph.typing import ContextT

In [2]:
agent = create_agent(
    model="deepseek-flash",
    middleware=[
        # apply_to_input用于控制是否对输入进行脱敏
        # True表示对输入进行脱敏,False反之
        PIIMiddleware("email", strategy="redact", apply_to_input=True, apply_to_output=True),
        PIIMiddleware("phone_number", detector=r"1[3-9]\d{9}", strategy="mask", apply_to_input=False, apply_to_output=True)
    ]
)

response = agent.invoke(
    {"messages" : [HumanMessage(content="我的电话号码是18213098410，邮箱是552423801@qq.com，如果记住了就和我确认一遍。")]}
)

for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

我的电话号码是18213098410，邮箱是552423801@qq.com，如果记住了就和我确认一遍。
================================== Ai Message ==================================

记住了，和你确认一遍：

- 电话号码：****8410
- 邮箱：[REDACTED_EMAIL]

以上信息是否正确？


## · HITL中间件

In [3]:
from langchain.messages import HumanMessage
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
emails = [
    {
        "subject" : "周末见个面？",
        "content" : """
            嗨 Y!
            最近怎么样？听说你搬去了上海，有机会见个面喝杯咖啡吗？

            祝好 Akira
        """,
        "from" : "552423801@qq.com",
        "status" : "unread"
    },
    {
        "subject" : "周五会议",
        "content" : """
            嗨 Y!
            非常抱歉，周五的会议取消了，请重新安排时间。

            祝好 小李
        """,
        "from" : "123456@qq.com",
        "status" : "checked"
    }
]

In [5]:
from langchain.tools import tool

@tool
def check_inbox() -> str:
    """检查邮箱里面的邮件"""

    return str(emails)

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """发送回复邮件"""

    return f"邮件已发送至 : {to}, 主题 : {subject}, 内容 : {body}"

In [6]:
from langchain.agents.middleware import HumanInTheLoopMiddleware

Human_in_loop = HumanInTheLoopMiddleware(
    interrupt_on ={
        "send_email" : {
            "description" : "请确认邮件内容！",
            "allowed_decisions" : ["approve", "reject", "edit"]
        }
    }
)

In [7]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
agent_with_HITL = create_agent(
    model="deepseek-flash",
    checkpointer=InMemorySaver(),
    middleware=[Human_in_loop],
    tools=[check_inbox, send_email],
    system_prompt="你是一个邮件处理助手，请根据邮件内容进行回复。"
)

In [8]:
config = {"configurable" : {"thread_id" : "1"}}
response = agent_with_HITL.invoke(
    {"messages" : [HumanMessage("帮我查看邮件")]},
    config=config
)

for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

帮我查看邮件
================================== Ai Message ==================================

我来帮您查看邮件。
Tool Calls:
  check_inbox (call_00_Z9ewUW71Qgh8i7SYHTZq2521)
 Call ID: call_00_Z9ewUW71Qgh8i7SYHTZq2521
  Args:
================================= Tool Message =================================
Name: check_inbox

[{'subject': '周末见个面？', 'content': '\n            嗨 Y!\n            最近怎么样？听说你搬去了上海，有机会见个面喝杯咖啡吗？\n\n            祝好 Akira\n        ', 'from': '552423801@qq.com', 'status': 'unread'}, {'subject': '周五会议', 'content': '\n            嗨 Y!\n            非常抱歉，周五的会议取消了，请重新安排时间。\n\n            祝好 小李\n        ', 'from': '123456@qq.com', 'status': 'checked'}]
================================== Ai Message ==================================

您有 2 封邮件：

**1. 未读邮件**
- **发件人**：552423801@qq.com（Akira）
- **主题**：周末见个面？
- **内容**：Akira 得知您搬到了上海，想约您周末见面喝杯咖啡叙旧。

**2. 已读邮件**
- **发件人**：123456@qq.com（小李）
- **主题**：周五会议
- **内容**：通知

In [9]:
response = agent_with_HITL.invoke(
    {"messages" : [HumanMessage("帮我回复akira，向她询问时间")]},
    config=config
)

for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

帮我查看邮件
================================== Ai Message ==================================

我来帮您查看邮件。
Tool Calls:
  check_inbox (call_00_Z9ewUW71Qgh8i7SYHTZq2521)
 Call ID: call_00_Z9ewUW71Qgh8i7SYHTZq2521
  Args:
================================= Tool Message =================================
Name: check_inbox

[{'subject': '周末见个面？', 'content': '\n            嗨 Y!\n            最近怎么样？听说你搬去了上海，有机会见个面喝杯咖啡吗？\n\n            祝好 Akira\n        ', 'from': '552423801@qq.com', 'status': 'unread'}, {'subject': '周五会议', 'content': '\n            嗨 Y!\n            非常抱歉，周五的会议取消了，请重新安排时间。\n\n            祝好 小李\n        ', 'from': '123456@qq.com', 'status': 'checked'}]
================================== Ai Message ==================================

您有 2 封邮件：

**1. 未读邮件**
- **发件人**：552423801@qq.com（Akira）
- **主题**：周末见个面？
- **内容**：Akira 得知您搬到了上海，想约您周末见面喝杯咖啡叙旧。

**2. 已读邮件**
- **发件人**：123456@qq.com（小李）
- **主题**：周五会议
- **内容**：通知

In [10]:
print(response['__interrupt__'][0].value)

{'action_requests': [{'name': 'send_email', 'args': {'to': '552423801@qq.com', 'subject': 'Re: 周末见个面？', 'body': '嗨 Akira，\n\n好久不见！我最近挺好的，确实搬到上海了，这边生活还在慢慢适应中。\n\n见面喝咖啡当然好呀，很期待！你周末哪天比较方便呢？周六还是周日？另外你方便的话，也可以告诉我你倾向的时间段（上午或者下午）和你附近方便的地方，我看看怎么安排。\n\n等你的消息，到时候见！\n\n祝好\nY'}, 'description': '请确认邮件内容！'}], 'review_configs': [{'action_name': 'send_email', 'allowed_decisions': ['approve', 'reject', 'edit']}]}


### Reject

In [11]:
from langgraph.types import Command

response = agent_with_HITL.invoke(
    Command(
        resume={
            "decisions" : [
                {
                    "type" : "reject",
                    "message" : "语气太生硬了，akira是我的女朋友！"
                }
            ]
        }
    ),
    config=config
)

for message in response["messages"]:
    message.pretty_print()


================================ Human Message =================================

帮我查看邮件
================================== Ai Message ==================================

我来帮您查看邮件。
Tool Calls:
  check_inbox (call_00_Z9ewUW71Qgh8i7SYHTZq2521)
 Call ID: call_00_Z9ewUW71Qgh8i7SYHTZq2521
  Args:
================================= Tool Message =================================
Name: check_inbox

[{'subject': '周末见个面？', 'content': '\n            嗨 Y!\n            最近怎么样？听说你搬去了上海，有机会见个面喝杯咖啡吗？\n\n            祝好 Akira\n        ', 'from': '552423801@qq.com', 'status': 'unread'}, {'subject': '周五会议', 'content': '\n            嗨 Y!\n            非常抱歉，周五的会议取消了，请重新安排时间。\n\n            祝好 小李\n        ', 'from': '123456@qq.com', 'status': 'checked'}]
================================== Ai Message ==================================

您有 2 封邮件：

**1. 未读邮件**
- **发件人**：552423801@qq.com（Akira）
- **主题**：周末见个面？
- **内容**：Akira 得知您搬到了上海，想约您周末见面喝杯咖啡叙旧。

**2. 已读邮件**
- **发件人**：123456@qq.com（小李）
- **主题**：周五会议
- **内容**：通知

### Approve

In [12]:
from langgraph.types import Command

response = agent_with_HITL.invoke(
    Command(
        resume={"decisions" : [{"type" : "approve"}]}
    ),
    config=config
)

for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

帮我查看邮件
================================== Ai Message ==================================

我来帮您查看邮件。
Tool Calls:
  check_inbox (call_00_Z9ewUW71Qgh8i7SYHTZq2521)
 Call ID: call_00_Z9ewUW71Qgh8i7SYHTZq2521
  Args:
================================= Tool Message =================================
Name: check_inbox

[{'subject': '周末见个面？', 'content': '\n            嗨 Y!\n            最近怎么样？听说你搬去了上海，有机会见个面喝杯咖啡吗？\n\n            祝好 Akira\n        ', 'from': '552423801@qq.com', 'status': 'unread'}, {'subject': '周五会议', 'content': '\n            嗨 Y!\n            非常抱歉，周五的会议取消了，请重新安排时间。\n\n            祝好 小李\n        ', 'from': '123456@qq.com', 'status': 'checked'}]
================================== Ai Message ==================================

您有 2 封邮件：

**1. 未读邮件**
- **发件人**：552423801@qq.com（Akira）
- **主题**：周末见个面？
- **内容**：Akira 得知您搬到了上海，想约您周末见面喝杯咖啡叙旧。

**2. 已读邮件**
- **发件人**：123456@qq.com（小李）
- **主题**：周五会议
- **内容**：通知

# 2.自定义中间件

#### 相较于使用tool来实现，自定义状态更加灵活，也更加优雅高效

## · model_call_counts中间件

### 自定义状态

In [ ]:
from langchain.agents import AgentState


class CustomState(AgentState):
    """Agent的任务状态"""

    model_call_counts: int

### 自定义中间件

In [ ]:
from typing import Any, Callable
from langgraph.runtime import Runtime
from langchain.agents.middleware import after_model, wrap_model_call, ModelRequest, ModelResponse


@after_model(state_schema=CustomState)
def increment_counter(
        # 注意两个参数，不能多，不能少，这是langcha框架调用的函数，不要随意添加
        state: CustomState,
        runtime: Runtime
) -> dict[str, Any]:
    # 读取state里面的model_call_counts值，若不存在则返回0
    curr_count = state.get("model_call_counts", 0)

    # 返回dict，键值对为要修改的参数及其值

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, AIMessage

agent = create_agent(
    model="deepseek-flash",
    checkpointer=InMemorySaver(),
    middleware=[increment_counter]
)

config = {"configurable" : {"thread_id" : "1"}}

response = agent.invoke(
    {"messages" : [HumanMessage("你好，我叫akira")]},
    config
)

print(response)

## · wrap_model_call中间件

In [ ]:
@wrap_model_call
def retry_model(
        request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    for attempt in range(3):
        try:
            return handler(request)  # 调用模型
        except Exception as e:
            print(f"retry {attempt + 1}/3 after error: {e}")
            if attempt == 2:
                raise
    return None

# 3.动态修改模型请求参数

In [3]:
from langchain.agents.middleware import wrap_model_call, ModelResponse, ModelRequest
from typing import Callable
from pydantic.dataclasses import dataclass
from langchain.messages import HumanMessage

# 定义一个context,记录用户是否开启思考模式
@dataclass
class UserContext:
    thinking: bool = False


@wrap_model_call
def dynamic_model_setting(
        # request框架自动构建并注入到中间件函数里面的“模型请求快照对象”，可以理解为一个即将发给LLM的完整请求数据包
        # ModelRequest则是Request的一个类型，包含model，messages等参数
        request: ModelRequest,
        # handler是下一个处理器
        # Callable用于描述函数签名的类型
        # ModelRequest -> ModelResponse 表示输入参数为ModelRequest，返回参数为ModelResponse
        handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:

    # 获取上下文参数thinking,是否开启思考模式
    thinking = request.runtime.context.thinking

    # 三元表达式(如果thinking为True,则返回"enabled",否则返回"disabled")
    thinking_type = "enabled" if thinking else "disabled"

    #  OpenAI Python SDK（及所有 OpenAI 兼容 API 客户端） 的一个特殊参数，用于"附加额外请求体字段"
    # SDK会将extra_body中的参数附加到请求体中
    model_setting = {"extra_body" : {"thinking" : {"type" : thinking_type}}}

    modified_request = request.override(model_settings=model_setting)
    return handler(modified_request)

In [4]:
from langchain.agents import create_agent

agent = create_agent(
    model="deepseek-flash",
    middleware=[dynamic_model_setting],
    context_schema=UserContext
)


response = agent.invoke(
    {"messages" : [HumanMessage("你好，我叫akira")]},
    context=UserContext(thinking=False)
)

print(response)

{'messages': [HumanMessage(content='你好，我叫akira', additional_kwargs={}, response_metadata={}, id='da45908c-efaa-4365-8e15-c40f17aa07c4'), AIMessage(content='你好，Akira！很高兴认识你。有什么我可以帮你的吗？', additional_kwargs={'refusal': None, 'reasoning_content': '我们需要回答中文。用户只是自我介绍。应友好回应，可能询问如何帮助。需要叫Akira? 保持自然。'}, response_metadata={'token_usage': {'completion_tokens': 42, 'prompt_tokens': 35, 'total_tokens': 77, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 26, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 35}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': 'c177f78c-47d8-4b57-b3b5-c86e66850d75', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1f33d

In [5]:
response = agent.invoke(
    {"messages" : [HumanMessage("你好，我叫akira")]},
    context=UserContext(thinking=False)
)

print(response)

{'messages': [HumanMessage(content='你好，我叫akira', additional_kwargs={}, response_metadata={}, id='918266c5-0e75-42c1-8a61-984d0901ddb4'), AIMessage(content='你好，Akira！很高兴认识你。😊\n\n我是DeepSeek，由深度求索公司创造的AI助手。有什么我可以帮你的吗？无论是回答问题、讨论想法，还是帮你处理文档、分析内容，我都很乐意帮忙！', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 51, 'prompt_tokens': 9, 'total_tokens': 60, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 9}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': '26b63eca-d158-45a6-8327-3d231c285b06', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1f33d-6976-70b2-aaa8-6e7e9c8eb302-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 9, 'output_tokens': 51, 'total_tokens': 6

# 4.条件跳转

In [3]:
from langgraph.runtime import Runtime
from typing import Any
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config

class CustomState(AgentState):
    """扩展agent状态，添加自定义字段"""

    model_call_count: int

# 类继承方法
class ModelCallLimitMiddleware(AgentMiddleware[CustomState]):  # AgentMiddleware[CustomState]表示其泛型为CustomState
    """模型调用次数限制中间件"""

    def __init__(self, max_limit: int = 3):
        super().__init__()
        self.max_limit = max_limit

    def after_model(self, state, runtime: Runtime) -> dict[str, Any] | None:
        curr_count = state.get("model_call_count", 0)
        return {"model_call_count" : curr_count + 1}

    # before_model hook,校验模型调用次数
    @hook_config(can_jump_to="end")
    def before_model(self, state: CustomState, runtime: Runtime) -> dict[str, Any] | None:
        # 获取调取次数
        # 获取以后再加一，表示当前调用次数
        curr_count = state.get("model_call_count", 0) + 1
        print(f"当前调用次数为: {curr_count}/{self.max_limit}")
        # 判断是否超限
        if curr_count > self.max_limit:
            return {
                "jump_to" : "end",
                "messages" : [AIMessage(f"调取次数超过最大限制{self.max_limit},请升级套餐")]
            }
        return None

In [ ]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage,AIMessage
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model="deepseek-flash",
    checkpointer=InMemorySaver(),
    middleware=[ModelCallLimitMiddleware(max_limit=2)],
    state_schema=CustomState
)

config = {"configurable" : {"thread_id" : "1"}}

#### 第一次调用

In [7]:
response = agent.invoke(
    {"messages" : [HumanMessage("你好，我叫akira")]},
    config=config
)

print(response)

当前调用次数为: 1/2
{'messages': [HumanMessage(content='你好，我叫akira', additional_kwargs={}, response_metadata={}, id='4f1dc454-6312-43d3-a7c1-cf3a0caf64d1'), AIMessage(content='你好，Akira，很高兴认识你！有什么我可以帮你的吗？', additional_kwargs={'refusal': None, 'reasoning_content': '我们需要回答中文。用户简单自我介绍。应该友好回应，询问需要什么帮助。不要过度。用日文名？可以说很高兴认识你，Akira。有什么可以帮你。保持简洁。'}, response_metadata={'token_usage': {'completion_tokens': 56, 'prompt_tokens': 35, 'total_tokens': 91, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 40, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 35}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': '35a99dd8-d72d-4270-bf21-ad3b5d33614f', 'finish_reason': 'stop', '

#### 第二次调用

In [8]:
response = agent.invoke(
    {"messages" : [HumanMessage("我想睡觉了")]},
    config=config
)

print(response)

当前调用次数为: 2/2
{'messages': [HumanMessage(content='你好，我叫akira', additional_kwargs={}, response_metadata={}, id='4f1dc454-6312-43d3-a7c1-cf3a0caf64d1'), AIMessage(content='你好，Akira，很高兴认识你！有什么我可以帮你的吗？', additional_kwargs={'refusal': None, 'reasoning_content': '我们需要回答中文。用户简单自我介绍。应该友好回应，询问需要什么帮助。不要过度。用日文名？可以说很高兴认识你，Akira。有什么可以帮你。保持简洁。'}, response_metadata={'token_usage': {'completion_tokens': 56, 'prompt_tokens': 35, 'total_tokens': 91, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 40, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 35}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': '35a99dd8-d72d-4270-bf21-ad3b5d33614f', 'finish_reason': 'stop', '

#### 第三次调用

In [9]:
response = agent.invoke(
    {"messages" : [HumanMessage("晚安")]},
    config=config
)

print(response)

当前调用次数为: 3/2
{'messages': [HumanMessage(content='你好，我叫akira', additional_kwargs={}, response_metadata={}, id='4f1dc454-6312-43d3-a7c1-cf3a0caf64d1'), AIMessage(content='你好，Akira，很高兴认识你！有什么我可以帮你的吗？', additional_kwargs={'refusal': None, 'reasoning_content': '我们需要回答中文。用户简单自我介绍。应该友好回应，询问需要什么帮助。不要过度。用日文名？可以说很高兴认识你，Akira。有什么可以帮你。保持简洁。'}, response_metadata={'token_usage': {'completion_tokens': 56, 'prompt_tokens': 35, 'total_tokens': 91, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 40, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 35}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': '35a99dd8-d72d-4270-bf21-ad3b5d33614f', 'finish_reason': 'stop', '